Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import seaborn as sns
from scipy import stats

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms

In [ ]:
galapagos = pd.read_csv("dados/galapagos.csv")
galapagos.head()

In [ ]:
model = smf.ols("Species ~ Area + Elevation + Nearest + Scruz + Adjacent", data=galapagos).fit()

In [ ]:
model.summary()

$$RSS = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

In [ ]:
sms.anova_lm(model)

## Álgebra Matricial

A partir daqui reproduzimos, passo a passo, os cálculos matriciais do script R.

### Montagem de $X$, $y$ e da matriz chapéu $H$

$$H = X(X^\top X)^{-1}X^\top$$

$H$ é a **matriz chapéu** (*hat matrix*): ela "coloca o chapéu" em $y$, pois $\hat{y} = Hy$.

In [ ]:
n = len(galapagos)           # número de observações
k = 5                        # número de preditores (sem contar o intercepto)

# Matriz de design X (n x 6): coluna de 1s + 5 preditores
# equivalente a: x = matrix(c(rep(1,n), gala$Area, ...), nrow=n, ncol=6)
X = np.column_stack([
    np.ones(n),
    galapagos["Area"],
    galapagos["Elevation"],
    galapagos["Nearest"],
    galapagos["Scruz"],
    galapagos["Adjacent"]
])

# Vetor resposta y
y = galapagos["Species"].values

# Matriz chapéu H = X (X'X)^{-1} X'
H = X @ np.linalg.inv(X.T @ X) @ X.T

print("Dimensões de X:", X.shape)
print("Dimensões de H:", H.shape)
print("\nPrimeiras linhas de X:")
print(X[:5])

### Soma de Quadrados dos Resíduos (SQRes) e estimativa de $\sigma^2$

$$SQRes = y^\top(I - H)y$$

$$\hat{\sigma}^2 = \frac{SQRes}{n - k - 1}$$

In [ ]:
# Matriz identidade n x n
In = np.eye(n)

# SQRes = y'(I - H)y
SQRes = y @ (In - H) @ y
print(f"SQRes = {SQRes:.4f}")       # equivale a deviance(model) no R

# Estimativa da variância dos erros: sigma^2
sigma2h = SQRes / (n - k - 1)
print(f"\nσ²  = {sigma2h:.4f}")
print(f"σ   = {np.sqrt(sigma2h):.4f}")

### Soma de Quadrados da Regressão (SQReg) e Teste F Global

$$SQReg = y^\top\left(H - \frac{J}{n}\right)y \quad \text{onde } J = \mathbf{1}\mathbf{1}^\top$$

A matriz $J/n$ centraliza $y$ em torno de $\bar{y}$, de modo que $SQReg$ mede o quanto a regressão explica além da média.

A estatística $F$ global testa $H_0: \beta_1 = \cdots = \beta_k = 0$:

$$F = \frac{SQReg / k}{SQRes / (n - k - 1)}$$

In [ ]:
# Matriz J: todos os elementos iguais a 1 (n x n)
J = np.ones((n, n))

# SQReg = y'(H - J/n)y
SQReg = y @ (H - J / n) @ y
print(f"SQReg = {SQReg:.4f}")

# Estatística F global
F_global = (SQReg / k) / (SQRes / (n - k - 1))
p_valor = 1 - stats.f.cdf(F_global, dfn=k, dfd=n - k - 1)

print(f"\nF     = {F_global:.4f}")
print(f"p-valor = {p_valor:.6f}")

### Estimativa dos coeficientes $\hat{\beta}$ e seus erros padrão

$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$

A matriz de covariâncias de $\hat{\beta}$:

$$\text{Var}(\hat{\beta}) = (X^\top X)^{-1} \hat{\sigma}^2$$

O erro padrão de cada $\hat{\beta}_j$ é a raiz quadrada do $j$-ésimo elemento da diagonal.

In [ ]:
# Coeficientes estimados: beta_hat = (X'X)^{-1} X'y
XtX_inv = np.linalg.inv(X.T @ X)
betah = XtX_inv @ X.T @ y

# Matriz de covariâncias de beta_hat
varbeta = XtX_inv * sigma2h

# Erros padrão (raiz da diagonal da matriz de covariâncias)
sdbeta = np.sqrt(np.diag(varbeta))

nomes = ["Intercepto", "Area", "Elevation", "Nearest", "Scruz", "Adjacent"]
print(f"{'Coef':<12} {'beta_hat':>12} {'std_err':>12}")
print("-" * 38)
for nome, b, s in zip(nomes, betah, sdbeta):
    print(f"{nome:<12} {b:>12.4f} {s:>12.4f}")

### Estatística $t$ para o intercepto

$$t_{\beta_0} = \frac{\hat{\beta}_0}{\widehat{\text{dp}}(\hat{\beta}_0)}$$

In [ ]:
tbeta0 = betah[0] / sdbeta[0]
print(f"t do intercepto = {tbeta0:.4f}")

# Confirmação via coeficientes do modelo smf
tbeta0_check = model.params["Intercept"] / sdbeta[0]
print(f"Confirmação     = {tbeta0_check:.4f}")

---
## ANOVA Sequencial (Tipo I)

A ANOVA sequencial decompõe $SQReg$ variável por variável.
A contribuição de cada variável é obtida pela **diferença** entre $SQReg$ dos modelos aninhados:

$$SQ(\beta_j \mid \beta_0, \ldots, \beta_{j-1}) = SQReg_{j} - SQReg_{j-1}$$

### Modelo com só Area

In [ ]:
# x1: intercepto + Area
x1 = np.column_stack([np.ones(n), galapagos["Area"]])
beta1h = np.linalg.inv(x1.T @ x1) @ x1.T @ y
H1 = x1 @ np.linalg.inv(x1.T @ x1) @ x1.T
SQReg1 = y @ (H1 - J / n) @ y

print(f"SQReg(Area) = {SQReg1:.4f}")

### Modelo com Area + Elevation

In [ ]:
# x2: intercepto + Area + Elevation
x2 = np.column_stack([np.ones(n), galapagos["Area"], galapagos["Elevation"]])
beta2h = np.linalg.inv(x2.T @ x2) @ x2.T @ y
H2 = x2 @ np.linalg.inv(x2.T @ x2) @ x2.T
SQReg2 = y @ (H2 - J / n) @ y

print(f"SQReg(Area + Elevation)              = {SQReg2:.4f}")
print(f"SQ(Elevation | Area) = SQReg2-SQReg1 = {SQReg2 - SQReg1:.4f}")

### Modelo com Area + Elevation + Nearest

In [ ]:
# x3: intercepto + Area + Elevation + Nearest
x3 = np.column_stack([np.ones(n), galapagos["Area"], galapagos["Elevation"], galapagos["Nearest"]])
beta3h = np.linalg.inv(x3.T @ x3) @ x3.T @ y
H3 = x3 @ np.linalg.inv(x3.T @ x3) @ x3.T
SQReg3 = y @ (H3 - J / n) @ y

print(f"SQReg(Area + Elevation + Nearest)             = {SQReg3:.4f}")
print(f"SQ(Nearest | Area, Elevation) = SQReg3-SQReg2 = {SQReg3 - SQReg2:.4f}")

---
## Teste de Hipótese Geral: $H_0: C\beta = m$

O teste F geral usa a **matriz de contrastes** $C$ para testar hipóteses lineares sobre $\beta$.

$$F = \frac{(C\hat{\beta} - m)^\top \left[C(X^\top X)^{-1}C^\top\right]^{-1}(C\hat{\beta} - m) \;/\; q}{SQRes/(n-k-1)}$$

onde $q$ é o número de restrições (linhas de $C$).

### Teste 1: $H_0: \beta_1 = \beta_2 = \beta_3 = \beta_4 = \beta_5 = 0$

In [ ]:
# Matriz C seleciona os 5 coeficientes de preditores (excluindo intercepto)
# C é (5 x 6): cada linha "aponta" para um beta
q = 5
C = np.array([
    [0, 1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1]
], dtype=float)

m = np.zeros(q)   # hipótese nula: cada beta = 0

diff = C @ betah - m
mid  = np.linalg.inv(C @ XtX_inv @ C.T)

num = diff @ mid @ diff
F_hip1 = (num / q) / (SQRes / (n - k - 1))
p_hip1  = 1 - stats.f.cdf(F_hip1, dfn=q, dfd=n - k - 1)

print(f"F = {F_hip1:.4f}")
print(f"p-valor = {p_hip1:.6f}")

### Teste 2: $H_0: \beta_1 = \beta_2 = \beta_3 = \beta_4 = \beta_5$

Formulado como diferenças consecutivas: $\beta_1 - \beta_2 = 0$, $\beta_2 - \beta_3 = 0$, etc.

In [ ]:
# C codifica as diferenças: linha i testa beta_i = beta_{i+1}
q2 = 4
C2 = np.array([
    [0,  1, -1,  0,  0,  0],
    [0,  0,  1, -1,  0,  0],
    [0,  0,  0,  1, -1,  0],
    [0,  0,  0,  0,  1, -1]
], dtype=float)

m2 = np.zeros(q2)

diff2 = C2 @ betah - m2
mid2  = np.linalg.inv(C2 @ XtX_inv @ C2.T)

num2   = diff2 @ mid2 @ diff2
F_hip2 = (num2 / q2) / (SQRes / (n - k - 1))
p_hip2  = 1 - stats.f.cdf(F_hip2, dfn=q2, dfd=n - k - 1)

print(f"F = {F_hip2:.4f}")
print(f"p-valor = {p_hip2:.6f}")

---
## Modelos alternativos

### Modelo sem intercepto

In [ ]:
# '- 1' remove o intercepto, equivalente ao R: lm(Species ~ ... - 1, gala)
model2 = smf.ols("Species ~ Area + Elevation + Nearest + Scruz + Adjacent - 1", data=galapagos).fit()
model2.summary()

### Modelo só com intercepto (modelo nulo)

In [ ]:
# Equivalente a lm(Species ~ 1, gala) no R
model3 = smf.ols("Species ~ 1", data=galapagos).fit()
model3.summary()